In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

In [94]:
categories=pd.read_csv('../datasets/domens_categories.csv')
ranks=pd.read_csv('../datasets/ranked_top1000000.csv')
secondary_prices=pd.read_csv('../datasets/secondary_prices.csv')

Датасет с целевой переменной

In [95]:
ton_domens=pd.read_csv('../raw_datasets/ton_domains_history.csv').drop(['tx_hash','sale_type'],axis=1)

убираем .ton

In [96]:
ton_domens['domain_name']=ton_domens['domain_name'].map(lambda x: x.split('.')[0])

Так как это список продаж, один и тот же домен может встречаться несколько раз. Избавляемся от дубликатов, а актуальную цену берём по последней продаже 

In [97]:
ton_domens['tx_time'] = pd.to_datetime(ton_domens['tx_time'], unit='s')
ton_domens.sort_values(by='tx_time', inplace=True)
ton_domens = ton_domens.groupby(by='domain_name', as_index=False).agg(ton_price=('price_ton', 'last'))
ton_domens = ton_domens.rename(columns={'domain_name': 'domain'})
ton_domens

,domain,ton_price
0,0---------------------------------------------...,1.04
1,0---------0,1.46
2,0---------1,1.00
3,0--------o0,1.00
4,0--0,100.00
...,...,...
167108,zzzzzzzzzzzzzzzzzz,1.00
167109,zzzzzzzzzzzzzzzzzzzzz,1.00
167110,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz...,1.00
167111,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz...,1.00


Переводим TON домены в utf-8, чтобы соответствовать остальному датасету

In [98]:
success = 0
error = 0
def to_unicode(label: str):
    global success, error
    if label.startswith("xn--"):
        try:
            label = label.encode("ascii").decode("idna")
            success += 1
            return label
        except:
            error += 1
            return label
    return label
ton_domens["domain"] = ton_domens["domain"].apply(to_unicode)
print(success, error)

3805 420


Успешно декодировали >90% punycode доменов

Собираем итоговый датасет

In [99]:
ton_dataset = ton_domens.merge(categories, on='domain', how='left')
len(ton_dataset[ton_dataset['category'].notna()])

10224

Всего 3121 доменов размеченных по категориям

In [100]:
ton_dataset=ton_dataset.merge(ranks, on='domain', how='left')

In [101]:
secondary_prices.rename(columns={'domain_name':'domain'}, inplace=True)
main_dataset = secondary_prices.merge(ton_dataset, on='domain', how='left')
main_dataset

,domain,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,sol_reg_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank
0,\t478300,8.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,\tニドキング,8.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,46.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,,18.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,,6.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3714759,🫸🫸🫸🫸🫸🫸,8.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3714760,🫸🫸🫸🫸🫸🫸🫸,8.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3714761,𠮷野家,80.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3714762,𨳒你老母,935.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Добавим признаки has_digit, has_letter, has_huphen и length

In [102]:
main_dataset["has_digit"] = main_dataset["domain"].str.contains(r"\d").astype(int)
main_dataset["has_letter"] = main_dataset["domain"].str.contains(r"[A-Za-z]").astype(int)
main_dataset["has_hyphen"] = main_dataset["domain"].str.contains("-").astype(int)
main_dataset["length"] = main_dataset["domain"].str.len()
main_dataset

,domain,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,sol_reg_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank,has_digit,has_letter,has_hyphen,length
0,\t478300,8.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,7.0
1,\tニドキング,8.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,6.0
2,,46.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3.0
3,,18.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,4.0
4,,6.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3714759,🫸🫸🫸🫸🫸🫸,8.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,6.0
3714760,🫸🫸🫸🫸🫸🫸🫸,8.22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,7.0
3714761,𠮷野家,80.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,3.0
3714762,𨳒你老母,935.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,4.0


In [103]:
dataset_with_ton = main_dataset[main_dataset["ton_price"].notna()]
dataset_without_ton = main_dataset[main_dataset["ton_price"].isna()]
dataset_with_ton.to_csv("../datasets/dataset_main.csv", index=False)
dataset_without_ton.to_csv("../datasets/dataset_without_target_value.csv", index=False)
dataset_with_ton

,domain,eth_reg_price,eth_sales_count,eth_sales_volume_usd,eth_last_sale_price_usd,sol_reg_price,sol_sales_count,sol_sales_volume_usd,sol_last_sale_price_usd,ton_price,category,rank,has_digit,has_letter,has_hyphen,length
72121,-💎-,62.55,NaN,NaN,NaN,21.000002,4.0,56070.65980,5391.16000,1.0,NaN,NaN,0,0,1,3.0
73313,0--0,112.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,1,0,1,4.0
73389,0-0-0,67.04,NaN,NaN,NaN,19.112482,NaN,NaN,NaN,50.0,NaN,NaN,1,0,1,5.0
73394,0-0-0-0,3.69,NaN,NaN,NaN,19.416770,NaN,NaN,NaN,30.0,NaN,NaN,1,0,1,7.0
73396,0-0-0-0-0,25132.31,1.0,189.6984,189.6984,19.744905,NaN,NaN,NaN,10.0,NaN,NaN,1,0,1,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3714002,🫅🏾,NaN,NaN,NaN,NaN,750.000000,2.0,499.67248,347.09998,1.0,NaN,NaN,0,0,0,2.0
3714006,🫅🏿,NaN,NaN,NaN,NaN,750.000000,3.0,383.32249,152.57250,1.0,NaN,NaN,0,0,0,2.0
3714200,🫡,NaN,NaN,NaN,NaN,731.561300,NaN,NaN,NaN,20.0,NaN,NaN,0,0,0,1.0
3714667,🫶🏻,NaN,NaN,NaN,NaN,749.944640,1.0,17502.00000,17502.00000,1.0,NaN,NaN,0,0,0,2.0


Мы получили два датасета, один без размеченной целевой переменной, другой без нее. Первый датасет может быть использован для обучения и проверки модели, а второй будет использоваться для отображения потенциальной цены продажи нового домена на сайте webdom.market.
На